In [1]:
# ============================================================
# BranchyNet + Adaptive Computation — ResNet-50 on CIFAR-10
# FIXED VERSION
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json, tempfile
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
import random

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 50
NUM_CLASSES = 10
SAVE_PATH   = "branchynet_resnet50_cifar10_fixed.pth"

EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
BRANCH_WEIGHTS  = [0.2, 0.3, 0.5]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

# ── UTILITY FUNCTIONS ─────────────────────────────────────────
# FIX: single consistent disk_size function used by BOTH baseline
#      and branchynet — uses 1024**2 (true MiB)
def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def measure_inference(model, device, batch_size=128, runs=50):
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp, n):
        with torch.no_grad():
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                s = torch.cuda.Event(enable_timing=True)
                e = torch.cuda.Event(enable_timing=True)
                s.record()
                for _ in range(n):
                    m(inp)
                e.record()
                torch.cuda.synchronize()
                return s.elapsed_time(e) / 1000.0
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    single   = torch.randn(1, 3, 32, 32, device=device)
    single_s = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    batch   = torch.randn(batch_size, 3, 32, 32, device=device)
    batch_s = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    return {
        "single_img_ms"      : round(single_ms, 4),
        "batch128_ms"        : round(batch_ms, 4),
        "per_img_ms"         : round(batch_ms / batch_size, 6),
        "throughput_imgs_sec": round(batch_size * runs / batch_s, 2),
        "timing_method"      : "CUDA events" if use_cuda else "perf_counter",
    }

def compute_flops(model, device, input_size=(1, 3, 32, 32)):
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        N, C_out, H_out, W_out = out.shape
        C_in = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) \
                  else (module.kernel_size, module.kernel_size)
        total_flops[0] += 2 * N * C_out * H_out * W_out * (C_in // module.groups) * kH * kW

    def linear_hook(module, inp, out):
        N = inp[0].shape[0]
        total_flops[0] += 2 * N * module.in_features * module.out_features

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    with torch.no_grad():
        m(torch.randn(*input_size, device=device))
    for h in hooks:
        h.remove()
    return round(total_flops[0] / 1e9, 6)

# ── AUXILIARY BRANCH ──────────────────────────────────────────
class EarlyExitBranch(nn.Module):
    def __init__(self, input_channels, num_classes=10):
        super().__init__()
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(input_channels),
            nn.Linear(input_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.branch(x)

# ── BRANCHYNET MODEL ──────────────────────────────────────────
class BranchyResNet50(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        backbone = models.resnet50(pretrained=pretrained)
        backbone.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        backbone.maxpool = nn.Identity()
        backbone.fc      = nn.Linear(backbone.fc.in_features, num_classes)

        self.num_classes = num_classes
        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4
        self.avgpool = backbone.avgpool
        self.fc      = backbone.fc

        self.branch1 = EarlyExitBranch(256, num_classes)
        self.branch2 = EarlyExitBranch(512, num_classes)

    def forward(self, x):
        """Full forward — all exits computed. Used during training."""
        x = self.stem(x)
        x = self.layer1(x);  logits1 = self.branch1(x)
        x = self.layer2(x);  logits2 = self.branch2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)
        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, x, threshold=0.8):
        """
        TRUE per-sample early exit.

        Each sample is routed independently:
          - If confident at exit1 → stops after layer1 (for that sample)
          - If confident at exit2 → stops after layer2
          - Otherwise            → runs full network

        Unconfident samples are sliced and passed forward through
        remaining layers only — confident samples never touch deeper layers.

        Returns
        -------
        final_logits : Tensor (B, num_classes)
        exit_idx     : Tensor (B,)  0=exit1, 1=exit2, 2=final
        """
        B = x.size(0)
        device = x.device

        # Accumulators
        final_logits = torch.zeros(B, self.num_classes, device=device)
        exit_idx     = torch.full((B,), 2, dtype=torch.long, device=device)

        # Global indices of samples still being processed
        remaining_global = torch.arange(B, device=device)

        # ── Stem + Exit 1 ────────────────────────────────────
        x = self.stem(x)
        x = self.layer1(x)
        logits1 = self.branch1(x)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values

        done1 = conf1 >= threshold
        if done1.any():
            g_idx = remaining_global[done1]
            final_logits[g_idx] = logits1[done1]
            exit_idx[g_idx]     = 0

        # Keep only unresolved samples for deeper layers
        keep = ~done1
        if not keep.any():
            return final_logits, exit_idx

        # Slice feature map and index tracker to unresolved samples only
        x                = x[keep]
        remaining_global = remaining_global[keep]

        # ── Exit 2 ───────────────────────────────────────────
        x = self.layer2(x)
        logits2 = self.branch2(x)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values

        done2 = conf2 >= threshold
        if done2.any():
            g_idx = remaining_global[done2]
            final_logits[g_idx] = logits2[done2]
            exit_idx[g_idx]     = 1

        keep = ~done2
        if not keep.any():
            return final_logits, exit_idx

        x                = x[keep]
        remaining_global = remaining_global[keep]

        # ── Exit 3 (final) ────────────────────────────────────
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)

        final_logits[remaining_global] = logits3
        # exit_idx already = 2 for these samples

        return final_logits, exit_idx


model = BranchyResNet50(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (BranchyNet): {total_params:,}")

# ── DATA ─────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set = torchvision.datasets.CIFAR10(root='../data', train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='../data', train=False,
                                          download=True, transform=transform_test)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                            shuffle=True, num_workers=0,
                                            worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                            shuffle=False, num_workers=0)
print(f"Train: {len(train_set)} | Test: {len(test_set)}")

# ── TRAINING ──────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1,
                             momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def branchynet_loss(logits_list, labels, weights=BRANCH_WEIGHTS):
    return sum(w * criterion(l, labels) for w, l in zip(weights, logits_list))

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits1, logits2, logits3 = model(inputs)
        loss = branchynet_loss([logits1, logits2, logits3], labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += logits3.argmax(1).eq(labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate_standard(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, _, logits3 = model(inputs)
            correct += logits3.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total

best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("TRAINING — BranchyNet (3 exits)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_acc = evaluate_standard(model, test_loader)
    scheduler.step()

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy (final exit): {best_val_acc:.4f}")

# ── FULL EVALUATION (final exit only) ─────────────────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_final  = accuracy_score(all_labels, all_preds)
prec_final = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec_final  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1_final   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc_final:.4f}")
print(f"  Precision (macro) : {prec_final:.4f}")
print(f"  Recall    (macro) : {rec_final:.4f}")
print(f"  F1-score  (macro) : {f1_final:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION ───────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)

adaptive_results = []

for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []
    t_start = time.time()

    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())

    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc  = accuracy_score(labels_arr, preds_arr)
    prec = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec  = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1   = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc,  6),
        "precision"  : round(prec, 6),
        "recall"     : round(rec,  6),
        "f1"         : round(f1,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

# ── MODEL COMPLEXITY METRICS ──────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

flops_g = compute_flops(model, DEVICE)
infer   = measure_inference(model, DEVICE)
size_mb = disk_size_mb(model)   # FIX: consistent with baseline

# Adaptive timing at τ=0.8, single image, GPU events
use_cuda     = DEVICE.type == "cuda"
dummy_single = torch.randn(1, 3, 32, 32, device=DEVICE)
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        e.record()
        torch.cuda.synchronize()
        inf_ms_adapt = s.elapsed_time(e) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        inf_ms_adapt = (time.perf_counter() - t0) / 50 * 1000

print(f"  Parameters               : {total_params:,}")
print(f"  Model size               : {size_mb:.4f} MB")
print(f"  FLOPs (full)             : {flops_g:.4f} G")
print(f"  Inference (single, full) : {infer['single_img_ms']:.2f} ms")
print(f"  Inference (batch128)     : {infer['batch128_ms']:.2f} ms "
      f"({infer['throughput_imgs_sec']:.0f} img/s)")
print(f"  Inference (τ=0.8 adapt)  : {inf_ms_adapt:.3f} ms")

# ── SAVE METRICS JSON ─────────────────────────────────────────
best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

branchynet_metrics = {
    "method"                      : "early_exit",
    "variant"                     : "BranchyNet_ResNet50",
    "dataset"                     : "CIFAR-10",
    "accuracy"                    : round(acc_final,  6),
    "precision"                   : round(prec_final, 6),
    "recall"                      : round(rec_final,  6),
    "f1"                          : round(f1_final,   6),
    "params"                      : total_params,
    "size_mb"                     : size_mb,
    "flops_G"                     : flops_g,
    "inference_ms"                : infer,
    "num_exits"                   : 3,
    "exit_positions"              : ["after layer1 (256ch)",
                                     "after layer2 (512ch)",
                                     "final fc (2048ch)"],
    "branch_weights"              : BRANCH_WEIGHTS,
    "exit_thresholds_tested"      : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08" : round(inf_ms_adapt, 4),
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    "saved_model_path"            : os.path.abspath(SAVE_PATH),
}

output_json = "__3__branchynet_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchynet_metrics, f, indent=2)
print(f"\n✓ Metrics saved to {output_json}")

# ── COMPARISON SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("COMPARISON SUMMARY: Baseline vs BranchyNet")
print("="*60)

baseline_path = "__2__baseline_metrics.json"
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        base = json.load(f)

    scalar_keys = ["accuracy", "precision", "recall", "f1",
                   "params", "size_mb", "flops_G"]
    print(f"\n  {'Metric':<22} {'Baseline':>12} {'BranchyNet':>12} {'Δ':>10}")
    print("  " + "-"*58)
    for k in scalar_keys:
        bv = base.get(k, float('nan'))
        mv = branchynet_metrics.get(k, float('nan'))
        if not isinstance(bv, (int, float)) or not isinstance(mv, (int, float)):
            continue
        delta = mv - bv
        sign  = "+" if delta > 0 else ""
        if k == "params":
            print(f"  {k:<22} {bv:>12,} {mv:>12,} {sign}{delta:>9,.0f}")
        else:
            print(f"  {k:<22} {bv:>12.4f} {mv:>12.4f} {sign}{delta:>9.4f}")

    full_ms = infer['single_img_ms']
    print(f"\n  Inference (single, full)  : {full_ms:.2f} ms")
    print(f"  Inference (τ=0.8 adapt)   : {inf_ms_adapt:.3f} ms  "
          f"({(1 - inf_ms_adapt/full_ms)*100:+.1f}% vs full single)")
    print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
    print(f"    Accuracy : {best_adaptive['accuracy']:.4f}")
    print(f"    Exit1    : {best_adaptive['frac_exit1']:.1%}")
    print(f"    Exit2    : {best_adaptive['frac_exit2']:.1%}")
    print(f"    Exit3    : {best_adaptive['frac_exit3']:.1%}")
    print(f"    ms/sample: {best_adaptive['avg_time_ms']:.4f}")
else:
    print(f"  (baseline JSON not found at {baseline_path})")

# ── CHECKPOINT ────────────────────────────────────────────────
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "model_name"    : "BranchyResNet50_CIFAR10",
        "num_classes"   : NUM_CLASSES,
        "input_size"    : [3, 32, 32],
        "num_exits"     : 3,
        "branch_weights": BRANCH_WEIGHTS,
        "normalization" : {"mean": CIFAR_MEAN, "std": CIFAR_STD},
    },
    "classes": CIFAR10_CLASSES,
}, "__3__model_checkpoint.pth")

print("\n" + "="*60)
print("BRANCHYNET — COMPLETE")
print("="*60)
print(f"  Weights    → {SAVE_PATH}")
print(f"  Checkpoint → __3__model_checkpoint.pth")
print(f"  Metrics    → {output_json}")

Using device: cuda


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Total parameters (BranchyNet): 23,623,518


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 | Test: 10000

TRAINING — BranchyNet (3 exits)
Epoch  1/50 | Loss: 1.4231 | Train: 0.6804 | Val: 0.7856 ← best saved
Epoch  2/50 | Loss: 1.1011 | Train: 0.8314 | Val: 0.8002 ← best saved
Epoch  3/50 | Loss: 1.0331 | Train: 0.8516 | Val: 0.8124 ← best saved
Epoch  4/50 | Loss: 0.9972 | Train: 0.8577 | Val: 0.8479 ← best saved
Epoch  5/50 | Loss: 0.9720 | Train: 0.8640 | Val: 0.7696
Epoch  6/50 | Loss: 0.9576 | Train: 0.8666 | Val: 0.8063
Epoch  7/50 | Loss: 0.9457 | Train: 0.8691 | Val: 0.8150
Epoch  8/50 | Loss: 0.9364 | Train: 0.8700 | Val: 0.8335
Epoch  9/50 | Loss: 0.9250 | Train: 0.8738 | Val: 0.8175
Epoch 10/50 | Loss: 0.9174 | Train: 0.8745 | Val: 0.8397
Epoch 11/50 | Loss: 0.9120 | Train: 0.8774 | Val: 0.8001
Epoch 12/50 | Loss: 0.9025 | Train: 0.8806 | Val: 0.7470
Epoch 13/50 | Loss: 0.8958 | Train: 0.8826 | Val: 0.8113
Epoch 14/50 | Loss: 0.8879 | Train: 0.8849 | Val: 0.8127
Epoch 15/50 | Loss: 0.8814 | Train: 0.8893 | Val: 0.7996
Epoch 16/50 | Loss: 0.8728 | Trai